## **SNAP Jupyter demo notebook**
**Classical Sentinel-1 IW InSAR — wrapped interferogram, phase unwrapping, and line-of-sight surface displacement**

In summary, this workflow contains:

- The classical end-to-end Sentinel-1 IW InSAR chain in SNAP
- Part 1: produce the wrapped, debursted, topo-phase-removed, Goldstein-filtered interferogram
- Part 2: export the interferogram to SNAPHU format
- Part 3: run SNAPHU externally to unwrap the phase
- Part 4: import the unwrapped phase, convert to displacement, geocode

Complexity: advanced

##### ***About the test data:***

Sentinel-1 IW SLC pair (reference + secondary) covering the same area, same track, with a short temporal baseline. Download from [Copernicus Browser](https://dataspace.copernicus.eu/browser/). Filenames look like `S1A_IW_SLC__1SDV_*`.

Place both `.SAFE` directories under `data/` and update the path variables in the *Configure input paths* cell.

##### ***About SNAPHU:***

Phase unwrapping is done by [SNAPHU](https://web.stanford.edu/group/radar/softwareandlinks/sw/snaphu/), a separate command-line tool that SNAP does **not** bundle. Install it before running Part 3:

- **Linux / macOS**: build from source or install via your package manager (`apt install snaphu` on Debian/Ubuntu, `brew install snaphu` on macOS).
- **Windows**: there is no official Windows build. Use WSL (`apt install snaphu` inside WSL) or a community Windows binary.

Set the `SNAPHU_BIN` variable in the *Configure input paths* cell to the path of the `snaphu` executable.

##### ***Some information on the Python environment:***

In [ ]:
import os
import sys
print("Python version: " + sys.version)

import sysconfig
print("Location of esa_snappy package: " + sysconfig.get_paths()['purelib'] + os.sep + "esa_snappy")

##### ***Import Python packages...***

In [ ]:
import esa_snappy
from esa_snappy import ProductIO

import snapista
from snapista import Graph
from snapista import Operator

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

##### ***Convenience plot functions:***

In [ ]:
def _read_band(product, name):
    band = product.getBand(name)
    if band is None:
        raise KeyError(f"Band {name!r} not found. Available: {[b.getName() for b in product.getBands()]}")
    w, h = band.getRasterWidth(), band.getRasterHeight()
    data = np.zeros(w * h, np.float32)
    band.readPixels(0, 0, w, h, data)
    data.shape = h, w
    return data

def plot_band(product, name, title=None, cmap='viridis', vmin=None, vmax=None):
    data = _read_band(product, name)
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title or name)
    fig.colorbar(im, ax=ax)
    plt.show()

def plot_interferogram(product, phase_band_name, coh_band_name=None):
    phi = _read_band(product, phase_band_name)
    if coh_band_name is None:
        fig, ax = plt.subplots(figsize=(7, 6))
        im = ax.imshow(phi, cmap='hsv', vmin=-np.pi, vmax=np.pi)
        ax.set_title('Wrapped interferogram phase [rad]')
        fig.colorbar(im, ax=ax)
    else:
        coh = _read_band(product, coh_band_name)
        fig, axs = plt.subplots(1, 2, figsize=(13, 5))
        im0 = axs[0].imshow(phi, cmap='hsv', vmin=-np.pi, vmax=np.pi)
        axs[0].set_title('Wrapped phase [rad]')
        fig.colorbar(im0, ax=axs[0])
        im1 = axs[1].imshow(coh, cmap='viridis', vmin=0, vmax=1)
        axs[1].set_title('Coherence')
        fig.colorbar(im1, ax=axs[1])
    plt.show()

def find_band(product, *patterns):
    names = [b.getName() for b in product.getBands()]
    for pat in patterns:
        for n in names:
            if pat.lower() in n.lower():
                return n
    raise KeyError(f"No band matching {patterns!r} found. Available: {names}")

---

### ***Background: the classical IW InSAR chain***

```text
                Reference SLC                  Secondary SLC
                     │                              │
             Apply-Orbit-File              Apply-Orbit-File
                     │                              │
              TOPSAR-Split                  TOPSAR-Split
                     │                              │
                     └───── Back-Geocoding ─────┘
                              │
                  Enhanced-Spectral-Diversity
                              │
                      Interferogram
                  (flat-earth + coherence)
                              │
                      TOPSAR-Deburst
                              │
                    TopoPhaseRemoval
                              │
                  GoldsteinPhaseFiltering
                              │
                     [SnaphuExport]    → SNAPHU (external) → [SnaphuImport]
                                                                   │
                                                         PhaseToDisplacement
                                                                   │
                                                          Terrain-Correction
                                                                   │
                                                                Write
```

Each step in plain English:

1. **Apply-Orbit-File** — download Precise Orbit Ephemerides (POE) and replace the on-board orbit with sub-centimetre-accurate state vectors.
2. **TOPSAR-Split** — pick one subswath and a contiguous burst range.
3. **Back-Geocoding** — use a DEM to predict per-pixel offsets between reference and secondary, and resample the secondary onto the reference grid.
4. **Enhanced-Spectral-Diversity (ESD)** — iteratively refine the azimuth registration at burst overlaps (TOPS-specific).
5. **Interferogram** — compute the complex conjugate multiplication, subtract the flat-earth phase using the orbits and a reference ellipsoid, and produce the coherence band.
6. **TOPSAR-Deburst** — stitch bursts into one continuous subswath strip.
7. **TopoPhaseRemoval** — use the DEM to compute and subtract the topographic phase contribution, leaving (ideally) only the displacement signal.
8. **Goldstein filter** — reduce phase noise to make the next step (unwrapping) work better.
9. **SNAPHU export / unwrap / import** — take the wrapped phase, integrate it into an absolute phase field. This is the step that turns a fringe pattern into a continuous displacement map.
10. **PhaseToDisplacement** — multiply unwrapped phase by `-λ / (4π)` to convert from radians to metres of line-of-sight displacement.
11. **Terrain-Correction** — geocode to a map projection.

---

### ***Configure input paths***

In [ ]:
data_dir = os.path.join(os.getcwd(), 'data')
graphs_dir = os.path.join(os.getcwd(), 'graphs')
results_dir = os.path.join(os.getcwd(), 'results')
snaphu_dir = os.path.join(os.getcwd(), 'snaphu')
os.makedirs(graphs_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)
os.makedirs(snaphu_dir, exist_ok=True)

reference_slc = os.path.join(data_dir, 'S1A_IW_SLC__1SDV_<reference>.SAFE', 'manifest.safe')
secondary_slc = os.path.join(data_dir, 'S1A_IW_SLC__1SDV_<secondary>.SAFE', 'manifest.safe')

# Subswath + burst range
subswath = 'IW1'
polarisation = 'VV'
first_burst = '1'
last_burst = '3'

# Path to the snaphu executable (Part 3 only)
SNAPHU_BIN = 'snaphu'  # assumes snaphu is on PATH; otherwise put the full path

---
## ***Part 1 — Wrapped interferogram***
---

We build the full classical chain up to (and including) Goldstein filtering. The output is the wrapped interferogram — a beautiful fringe pattern but still modulo-2π.

In [ ]:
def build_iw_branch(g, slc_path, tag):
    g.add_node(operator=Operator('Read', file=slc_path),
               node_id=f'Read_{tag}')
    g.add_node(operator=Operator('Apply-Orbit-File',
                                 orbitType='Sentinel Precise (Auto Download)',
                                 continueOnFail='true'),
               node_id=f'Orbit_{tag}', source=f'Read_{tag}')
    g.add_node(operator=Operator('TOPSAR-Split',
                                 subswath=subswath,
                                 selectedPolarisations=polarisation,
                                 firstBurstIndex=first_burst,
                                 lastBurstIndex=last_burst),
               node_id=f'Split_{tag}', source=f'Orbit_{tag}')
    return f'Split_{tag}'

g_ifg = Graph()
ref_node = build_iw_branch(g_ifg, reference_slc, 'ref')
sec_node = build_iw_branch(g_ifg, secondary_slc, 'sec')

g_ifg.add_node(operator=Operator('Back-Geocoding',
                                 demName='Copernicus 30m Global DEM',
                                 resamplingType='BISINC_5_POINT_INTERPOLATION'),
               node_id='BackGeo', source=[ref_node, sec_node])
g_ifg.add_node(operator=Operator('Enhanced-Spectral-Diversity'),
               node_id='ESD', source='BackGeo')
g_ifg.add_node(operator=Operator('Interferogram',
                                 subtractFlatEarthPhase='true',
                                 subtractTopographicPhase='false',
                                 includeCoherence='true',
                                 cohWinAz='10',
                                 cohWinRg='10'),
               node_id='Ifg', source='ESD')
g_ifg.add_node(operator=Operator('TOPSAR-Deburst'),
               node_id='Deburst', source='Ifg')
g_ifg.add_node(operator=Operator('TopoPhaseRemoval',
                                 demName='Copernicus 30m Global DEM'),
               node_id='Topo', source='Deburst')
g_ifg.add_node(operator=Operator('GoldsteinPhaseFiltering', alpha='1.0'),
               node_id='Goldstein', source='Topo')

wrapped_out = os.path.join(results_dir, 'snap_nb_insar_wrapped.dim')
g_ifg.add_node(operator=Operator('Write', file=wrapped_out, formatName='BEAM-DIMAP'),
               node_id='Write', source='Goldstein')

g_ifg.save_graph(os.path.join(graphs_dir, 'snap_nb_insar_wrapped.xml'))
g_ifg.run()

In [ ]:
p_wrapped = ProductIO.readProduct(wrapped_out)
phase = find_band(p_wrapped, 'Phase_ifg', 'phase_')
coh = find_band(p_wrapped, 'coh_')
plot_interferogram(p_wrapped, phase, coh)
p_wrapped.dispose()

---
## ***Part 2 — Export to SNAPHU format***
---

`SnaphuExport` writes the wrapped phase, coherence and a `snaphu.conf` config file into a directory that SNAPHU can read directly. The exported phase/coh bands are byte-for-byte the inputs SNAPHU expects.

In [ ]:
g_export = Graph()
g_export.add_node(operator=Operator('Read', file=wrapped_out), node_id='Read')
g_export.add_node(operator=Operator('SnaphuExport',
                                    targetFolder=snaphu_dir,
                                    statCostMode='DEFO',
                                    initMethod='MCF',
                                    numberOfTileRows='1',
                                    numberOfTileCols='1',
                                    numberOfProcessors='4',
                                    rowOverlap='200',
                                    colOverlap='200'),
                  node_id='SnaphuExport', source='Read')

g_export.save_graph(os.path.join(graphs_dir, 'snap_nb_insar_snaphu_export.xml'))
g_export.run()

# Find the subdirectory SnaphuExport created
exported_subdirs = [d for d in os.listdir(snaphu_dir)
                    if os.path.isdir(os.path.join(snaphu_dir, d))]
print('SnaphuExport wrote into:', exported_subdirs)
snaphu_work_dir = os.path.join(snaphu_dir, exported_subdirs[0])
print('Working directory:', snaphu_work_dir)

---
## ***Part 3 — Run SNAPHU***
---

SNAPHU reads `snaphu.conf` from the working directory and writes an unwrapped `.img` file. We invoke it via `subprocess` here.

In [ ]:
import subprocess

# SnaphuExport writes snaphu.conf with the exact command line SNAPHU needs in a header comment.
conf_path = os.path.join(snaphu_work_dir, 'snaphu.conf')
with open(conf_path, 'r') as f:
    head = ''.join([next(f) for _ in range(20)])
print('--- snaphu.conf header (first 20 lines) ---')
print(head)

# Extract the suggested command line (looks like '# snaphu -f snaphu.conf <wrappedfile> <width>').
import re
match = re.search(r'^#\s*snaphu\s+(.+)$', head, re.MULTILINE)
if match is None:
    raise RuntimeError('Could not find suggested snaphu command line in snaphu.conf')
snaphu_args = match.group(1).split()
print('Will invoke:', [SNAPHU_BIN, *snaphu_args])

result = subprocess.run([SNAPHU_BIN, *snaphu_args],
                        cwd=snaphu_work_dir,
                        capture_output=True, text=True)
print('STDOUT:'); print(result.stdout[-1500:])
print('STDERR:'); print(result.stderr[-1500:])
if result.returncode != 0:
    raise RuntimeError(f'snaphu exited with code {result.returncode}')

---
## ***Part 4 — Import unwrapped phase, convert to displacement, geocode***
---

`SnaphuImport` reads the unwrapped `.img` SNAPHU produced and adds an `Unw_Phase_*` band back into the SNAP product. From there:

- **PhaseToDisplacement** scales the unwrapped phase to metres of line-of-sight (LOS) displacement using the radar wavelength.
- **Terrain-Correction** finally puts the displacement map in a map projection.

In [ ]:
g_disp = Graph()

# Read the wrapped-phase product (it still has the metadata SnaphuImport needs)
g_disp.add_node(operator=Operator('Read', file=wrapped_out), node_id='ReadWrapped')

# SnaphuImport pulls the .img unwrapped phase from the SnaphuExport working dir
g_disp.add_node(operator=Operator('SnaphuImport',
                                  doNotKeepWrapped='false'),
                node_id='SnaphuImport',
                source=['ReadWrapped'])
# Note: SnaphuImport expects the unwrapped .img to be in the same directory as the wrapped product's
# .data folder, OR you can pass the path through the source list. If your SNAP version requires the
# unwrapped file as a second product, read it first and pass both: source=['ReadWrapped', 'ReadUnw'].

g_disp.add_node(operator=Operator('PhaseToDisplacement'),
                node_id='Disp', source='SnaphuImport')

g_disp.add_node(operator=Operator('Terrain-Correction',
                                  demName='Copernicus 30m Global DEM',
                                  mapProjection='WGS84(DD)',
                                  pixelSpacingInMeter='20',
                                  imgResamplingMethod='BILINEAR_INTERPOLATION'),
                node_id='TC', source='Disp')

displacement_out = os.path.join(results_dir, 'snap_nb_insar_displacement.dim')
g_disp.add_node(operator=Operator('Write', file=displacement_out, formatName='BEAM-DIMAP'),
                node_id='Write', source='TC')

g_disp.save_graph(os.path.join(graphs_dir, 'snap_nb_insar_displacement.xml'))
g_disp.run()

In [ ]:
p_disp = ProductIO.readProduct(displacement_out)
print('Bands:', [b.getName() for b in p_disp.getBands()])

disp_band = find_band(p_disp, 'displacement', 'Displacement')
# Symmetric colour scale: subsidence (negative) red, uplift (positive) blue
data = _read_band(p_disp, disp_band)
vmax = np.nanpercentile(np.abs(data), 98)
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(data, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
ax.set_title('Line-of-sight displacement [m] — red = away from sensor, blue = toward')
fig.colorbar(im, ax=ax, label='m')
plt.show()
p_disp.dispose()

---

### ***Summary***

What have we learnt in this notebook?

- The classical Sentinel-1 IW InSAR chain has ≈10 GPF operators and one external tool (SNAPHU) for the unwrapping step.
- Phase unwrapping turns the wrapped (mod-2π) fringe pattern into an absolute phase field. SNAPHU's `DEFO` cost mode + `MCF` initialiser is a reasonable default for deformation studies.
- `PhaseToDisplacement` scales unwrapped phase by `-λ / (4π)` to give line-of-sight displacement in metres.
- The whole chain ends with `Terrain-Correction` so the displacement map is in a map projection ready for GIS.
- This is the *classical* path. The companion notebook [snap-nb-sar-gslc-insar](snap-nb-sar-gslc-insar.ipynb) shows the shorter GSLC alternative for the geocoded-complex case.